# 00 — Data Preprocessing (Configurable Filters)

Build a filtered dataset from the existing 16 hash-partitioned files.

## Configurable Parameters

| Parameter | Description |
|-----------|-------------|
| `VERSION` | Version string to append to output files (e.g. 'v2', 'v3') |
| `MIN_RATINGS` | Minimum ratings per user (drop users below) |
| `MAX_RATINGS` | Maximum ratings per user (`None` = no limit) |
| `MAX_USERS` | Max total users (`None` = all). Train and test share the same user set |
| `TEST_RATIO` | Fraction of each user's newest interactions for test (default 0.2) |

**Output Format:** `cb_train_interactions_{VERSION}.parquet`, `cb_books_{VERSION}.parquet`, etc.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import gc, os, psutil
from tqdm import tqdm

def print_ram(label=''):
    mem = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    tag = f' [{label}]' if label else ''
    print(f'  [RAM{tag}] {mem:.2f} GB')

# ============================================================
# CONFIGURABLE PARAMETERS
# ============================================================
VERSION     = 'v5'     # Version suffix for all output files
MIN_RATINGS = 5      # min ratings per user
MAX_RATINGS = None     # max ratings per user (None = no limit)
MAX_USERS   = 100     # max total users in BOTH train & test (None = all)
TEST_RATIO  = 0.2      # fraction newest interactions -> test

# ============================================================
# PATHS
# ============================================================
BASE_DIR      = '/content/drive/My Drive/Thesis/book_recsys'
DATA_DIR      = os.path.join(BASE_DIR, 'data/processed')
PARTITION_DIR = os.path.join(DATA_DIR, 'cb_partitions')
BOOKS_V1      = os.path.join(DATA_DIR, 'cb_books.parquet')

VALID_USERS_OUT = os.path.join(DATA_DIR, f'cb_valid_users_{VERSION}.parquet')
TRAIN_OUT       = os.path.join(DATA_DIR, f'cb_train_interactions_{VERSION}.parquet')
TEST_OUT        = os.path.join(DATA_DIR, f'cb_test_interactions_{VERSION}.parquet')
BOOKS_OUT       = os.path.join(DATA_DIR, f'cb_books_{VERSION}.parquet')

HEX_CHARS = '0123456789abcdef'

print('=' * 55)
print('  CONFIGURATION')
print('=' * 55)
print(f'  VERSION:     {VERSION}')
print(f'  MIN_RATINGS: {MIN_RATINGS}')
print(f'  MAX_RATINGS: {MAX_RATINGS if MAX_RATINGS else "No limit"}')
print(f'  MAX_USERS:   {MAX_USERS if MAX_USERS else "All"}')
print(f'  TEST_RATIO:  {TEST_RATIO}')
print('=' * 55)
print_ram()

  CONFIGURATION
  VERSION:     v5
  MIN_RATINGS: 5
  MAX_RATINGS: No limit
  MAX_USERS:   100
  TEST_RATIO:  0.2
  [RAM] 0.17 GB


## 0. Clean Previous Output for this Version

In [4]:
for path in [VALID_USERS_OUT, TRAIN_OUT, TEST_OUT, BOOKS_OUT]:
    if os.path.exists(path):
        os.remove(path)
        print(f'  Removed: {os.path.basename(path)}')
    else:
        print(f'  Not found (OK): {os.path.basename(path)}')
print('Ready for fresh build.')

  Not found (OK): cb_valid_users_v5.parquet
  Not found (OK): cb_train_interactions_v5.parquet
  Not found (OK): cb_test_interactions_v5.parquet
  Not found (OK): cb_books_v5.parquet
Ready for fresh build.


## 1. Find Valid Users (DuckDB)

In [5]:
!pip install -q duckdb

In [6]:
import duckdb
con = duckdb.connect()
glob_path = os.path.join(PARTITION_DIR, 'part_*.parquet')
total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{glob_path}')").fetchone()[0]
n_all = con.execute(f"SELECT COUNT(DISTINCT user_id) FROM read_parquet('{glob_path}')").fetchone()[0]
print(f'  Total interactions: {total:,}')
print(f'  Total users: {n_all:,}')
having = f'HAVING COUNT(*) >= {MIN_RATINGS}'
if MAX_RATINGS:
    having += f' AND COUNT(*) <= {MAX_RATINGS}'
con.execute(f"""
    COPY (
        SELECT user_id, COUNT(*) AS n_ratings
        FROM read_parquet('{glob_path}')
        GROUP BY user_id
        {having}
    ) TO '{VALID_USERS_OUT}' (FORMAT PARQUET)
""")
n_valid = con.execute(f"SELECT COUNT(*) FROM read_parquet('{VALID_USERS_OUT}')").fetchone()[0]
stats = con.execute(f"""
    SELECT MIN(n_ratings)::INT, MAX(n_ratings)::INT,
           AVG(n_ratings)::INT, MEDIAN(n_ratings)::INT
    FROM read_parquet('{VALID_USERS_OUT}')
""").fetchone()
con.close()
print(f'  Valid users: {n_valid:,} / {n_all:,} ({n_valid/n_all:.1%})')
print(f'  Ratings/user: min={stats[0]}, max={stats[1]}, avg={stats[2]}, median={stats[3]}')
print_ram('after DuckDB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Total interactions: 156,661,341
  Total users: 776,354


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Valid users: 743,367 / 776,354 (95.8%)
  Ratings/user: min=5, max=38884, avg=211, median=92
  [RAM [after DuckDB]] 0.21 GB


## 2. Select Final User Set

In [7]:
print('Loading valid user list...')
df_valid = pd.read_parquet(VALID_USERS_OUT)
user_list = df_valid['user_id'].tolist()
del df_valid
gc.collect()
print(f'  Users after rating filter: {len(user_list):,}')
if MAX_USERS and MAX_USERS < len(user_list):
    rng = np.random.default_rng(42)
    user_list = sorted(rng.choice(user_list, size=MAX_USERS, replace=False))
    print(f'  Sampled to MAX_USERS: {len(user_list):,}')
final_users = set(user_list)
del user_list
gc.collect()
print(f'  Final user set: {len(final_users):,} (shared by train & test)')
print_ram('after user selection')

Loading valid user list...
  Users after rating filter: 743,367
  Sampled to MAX_USERS: 100
  Final user set: 100 (shared by train & test)
  [RAM [after user selection]] 0.31 GB


## 3. Filter + Per-User Temporal Split

In [8]:
train_writer = None
test_writer  = None
n_train = 0
n_test  = 0
n_skipped = 0
for h in tqdm(HEX_CHARS, desc='Processing partitions'):
    pp = os.path.join(PARTITION_DIR, f'part_{h}.parquet')
    if not os.path.exists(pp):
        continue
    df = pd.read_parquet(pp)
    n_before = len(df)
    df = df[df['user_id'].isin(final_users)].copy()
    n_skipped += (n_before - len(df))
    if df.empty:
        del df
        gc.collect()
        continue
    df = df.sort_values(['user_id', 'timestamp'], ascending=[True, False])
    df['_rank']  = df.groupby('user_id').cumcount()
    df['_count'] = df.groupby('user_id')['user_id'].transform('count')
    df['_ntest'] = np.ceil(df['_count'] * TEST_RATIO).astype(int).clip(lower=1)
    test_mask  = df['_rank'] < df['_ntest']
    train_part = df[~test_mask][['user_id','book_id','rating','timestamp']].copy()
    test_part  = df[test_mask][['user_id','book_id','rating','timestamp']].copy()
    if not train_part.empty:
        t = pa.Table.from_pandas(train_part, preserve_index=False)
        if train_writer is None:
            train_writer = pq.ParquetWriter(TRAIN_OUT, t.schema)
        train_writer.write_table(t)
        n_train += len(train_part)
    if not test_part.empty:
        t = pa.Table.from_pandas(test_part, preserve_index=False)
        if test_writer is None:
            test_writer = pq.ParquetWriter(TEST_OUT, t.schema)
        test_writer.write_table(t)
        n_test += len(test_part)
    del df, train_part, test_part
    gc.collect()
if train_writer: train_writer.close()
if test_writer:  test_writer.close()
del final_users
gc.collect()
print(f'\nKept:    {n_train + n_test:,}')
print(f'Skipped: {n_skipped:,}')
print(f'Train:   {n_train:,}')
print(f'Test:    {n_test:,}')
print_ram('after split')

Processing partitions: 100%|██████████| 16/16 [00:59<00:00,  3.69s/it]


Kept:    10,044
Skipped: 104,333,032
Train:   7,995
Test:    2,049
  [RAM [after split]] 1.03 GB


## 4. Build `cb_books_{VERSION}.parquet`

In [9]:
print(f'Collecting book_ids from train + test {VERSION}...')
book_ids = set()
for path in [TRAIN_OUT, TEST_OUT]:
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=500_000, columns=['book_id']):
        book_ids.update(batch.column('book_id').to_pylist())
print(f'  Unique books: {len(book_ids):,}')
print(f'Filtering {os.path.basename(BOOKS_V1)}...')
writer = None
total = 0
pf = pq.ParquetFile(BOOKS_V1)
for batch in tqdm(pf.iter_batches(batch_size=50_000)):
    chunk = batch.to_pandas()
    filtered = chunk[chunk['book_id'].isin(book_ids)]
    if not filtered.empty:
        t = pa.Table.from_pandas(filtered, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(BOOKS_OUT, t.schema)
        writer.write_table(t)
        total += len(filtered)
    del chunk, filtered
    gc.collect()
if writer: writer.close()
del book_ids
gc.collect()
print(f'  Books {VERSION}: {total:,}')
print_ram('after books filtering')

  Unique books: 7,227
Filtering cb_books.parquet...


48it [00:39,  1.22it/s]


  Books v5: 7,227
  [RAM [after books filtering]] 0.95 GB


## 5. Summary

In [10]:
print('=' * 60)
print(f'  PREPROCESSING {VERSION} — SUMMARY')
print('=' * 60)
for name, path in [(f'cb_books_{VERSION}', BOOKS_OUT), (f'cb_train_{VERSION}', TRAIN_OUT),
                    (f'cb_test_{VERSION}', TEST_OUT), (f'cb_valid_users_{VERSION}', VALID_USERS_OUT)]:
    if os.path.exists(path):
        sz = os.path.getsize(path) / 1024**2
        n  = pq.read_metadata(path).num_rows
        print(f'  {name:<30} {n:>12,} rows  {sz:>8.1f} MB')
n_tr = pq.read_metadata(TRAIN_OUT).num_rows
n_te = pq.read_metadata(TEST_OUT).num_rows
print('-' * 60)
print(f'  Split ratio: {n_tr/(n_tr+n_te):.1%} train / {n_te/(n_tr+n_te):.1%} test')
print_ram('final')

  PREPROCESSING v5 — SUMMARY
  cb_books_v5                           7,227 rows       5.3 MB
  cb_train_v5                           7,995 rows       0.1 MB
  cb_test_v5                            2,049 rows       0.0 MB
  cb_valid_users_v5                   743,367 rows      23.6 MB
------------------------------------------------------------
  Split ratio: 79.6% train / 20.4% test
  [RAM [final]] 0.95 GB
